# AWS DynamoDB

## ClassicModels (Customers & Offices)

- DynamoDB is a **fully managed NoSQL** database provided by AWS, offering automatic scalability without manual intervention.

- Designed for **high availability and low-latency** performance, DynamoDB can handle millions of requests per second and scales horizontally to accommodate varying workloads.

- Supports **flexible data models** — unlike RDS there is no fixed schema per row. A new attribute can be added to a single item without altering a table definition.

- This notebook mirrors the **classicmodels** `customers` and `offices` tables from RDS, copies their rows into DynamoDB, and highlights the key differences between the two paradigms:

| Concept | RDS (MySQL) | DynamoDB |
|---------|-------------|----------|
| Schema | Fixed columns | Per-item (schemaless) |
| Relationships | Foreign keys / JOINs | Denormalise or use secondary indexes |
| Query language | SQL | Key-based + FilterExpression |
| Scaling | Vertical (+ read replicas) | Horizontal, automatic |
| Consistency | ACID transactions | Eventually consistent (default) |

https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/dynamodb.html


In [1]:
import boto3
import pymysql
import pymysql.cursors
import pandas as pd
from decimal import Decimal


## Connections

In [2]:
# Create a session using a specific profile
session = boto3.Session(profile_name='ashish')
credentials = session.get_credentials()

aws_access_key_id     = credentials.access_key
aws_secret_access_key = credentials.secret_key
region_name           = 'us-east-1'


In [3]:
# DynamoDB low-level client (used for table management)
dynamodb = boto3.client(
    'dynamodb',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region_name,
)

# DynamoDB resource (higher-level, used for put_item / query)
dynamodb_resource = boto3.resource(
    'dynamodb',
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region_name,
)

print("DynamoDB client and resource ready.")


DynamoDB client and resource ready.


## Create Tables – Customers & Offices

Two separate DynamoDB tables mirror the RDS `customers` and `offices` tables.  
Each table only needs the **key attributes** declared upfront — all other attributes are schemaless.

| Table | Partition Key | Sort Key |
|-------|--------------|----------|
| `ClassicModels_Customers` | `customerNumber` (N) | — |
| `ClassicModels_Offices`   | `officeCode` (S)     | — |


In [4]:
def create_table_if_not_exists(table_name, key_schema, attribute_definitions):
    """Create a DynamoDB table; skip if it already exists."""
    existing = dynamodb.list_tables()['TableNames']   # already a list of strings
    if table_name in existing:
        print(f"Table '{table_name}' already exists – skipping creation.")
        return

    dynamodb.create_table(
        TableName=table_name,
        KeySchema=key_schema,
        AttributeDefinitions=attribute_definitions,
        BillingMode='PAY_PER_REQUEST',   # on-demand – no capacity planning needed
    )
    waiter = dynamodb.get_waiter('table_exists')
    waiter.wait(TableName=table_name)
    print(f"Table '{table_name}' created successfully.")


# ── Customers table ──────────────────────────────────────────────────────────
create_table_if_not_exists(
    table_name='ClassicModels_Customers',
    key_schema=[{'AttributeName': 'customerNumber', 'KeyType': 'HASH'}],
    attribute_definitions=[{'AttributeName': 'customerNumber', 'AttributeType': 'N'}],
)

# ── Offices table ────────────────────────────────────────────────────────────
create_table_if_not_exists(
    table_name='ClassicModels_Offices',
    key_schema=[{'AttributeName': 'officeCode', 'KeyType': 'HASH'}],
    attribute_definitions=[{'AttributeName': 'officeCode', 'AttributeType': 'S'}],
)


Table 'ClassicModels_Customers' created successfully.
Table 'ClassicModels_Offices' created successfully.


## Connect to RDS (pymysql)

Open a pymysql connection to the RDS classicmodels database — this is the **source** for the copy.  
Update the credentials below to match your own RDS instance (same values as in `2.3-AWS-RDS.ipynb`).


In [5]:
# RDS endpoint  (console → RDS → Databases → your instance → Connectivity → Endpoint)
DB_HOST     = "database-1.cvkgi402g6iv.us-east-1.rds.amazonaws.com"
DB_PORT     = 3306
DB_USER     = "admin"           # master username set when you created the instance
DB_PASSWORD = "dzYNcYDfTI2hgz4FR6F3"   # store in Secrets Manager or an env var in production
DB_NAME     = "classicmodels"

rds_conn = pymysql.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
    cursorclass=pymysql.cursors.DictCursor,
    autocommit=True,
)
print("Connected to RDS:", rds_conn.get_server_info())


Connected to RDS: 8.4.8


In [6]:
def fetch_rds(sql):
    """Run a query on RDS and return a list of dicts."""
    with rds_conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchall()


def to_dynamo_types(row: dict) -> dict:
    """
    Convert Python types so they are safe for DynamoDB:
    - float / Decimal → Decimal  (DynamoDB rejects plain float)
    - None            → omitted  (DynamoDB has no NULL in put_item)
    """
    clean = {}
    for k, v in row.items():
        if v is None:
            continue                         # skip None values
        if isinstance(v, float):
            clean[k] = Decimal(str(v))
        else:
            clean[k] = v
    return clean


# ── Copy customers ──────────────────────────────────────────────────────────
customers_raw = fetch_rds("SELECT * FROM customers")
tbl_customers = dynamodb_resource.Table('ClassicModels_Customers')

with tbl_customers.batch_writer() as batch:
    for row in customers_raw:
        batch.put_item(Item=to_dynamo_types(row))

print(f"Copied {len(customers_raw)} customers to DynamoDB.")

# ── Copy offices ─────────────────────────────────────────────────────────────
offices_raw   = fetch_rds("SELECT * FROM offices")
tbl_offices   = dynamodb_resource.Table('ClassicModels_Offices')

with tbl_offices.batch_writer() as batch:
    for row in offices_raw:
        batch.put_item(Item=to_dynamo_types(row))

print(f"Copied {len(offices_raw)} offices to DynamoDB.")


Copied 122 customers to DynamoDB.
Copied 7 offices to DynamoDB.


## Read Data from DynamoDB

DynamoDB offers two read operations:

| Operation | When to use |
|-----------|-------------|
| `get_item` | Retrieve **one item** by its exact primary key — O(1) cost |
| `scan` | Read **every item** in the table — expensive, avoid on large tables |
| `query` | Read items sharing a **partition key** — fast and cheap |

Unlike SQL there is no `JOIN`. Relationships are handled by denormalisation or application-level lookups.


In [7]:
def scan_table(table_name: str) -> pd.DataFrame:
    """Scan an entire DynamoDB table and return a DataFrame (demo only – avoid on large tables)."""
    paginator = dynamodb.get_paginator('scan')
    items = []
    for page in paginator.paginate(TableName=table_name):
        items.extend(page['Items'])

    # Deserialize DynamoDB typed values {'S': '...'} → plain Python values
    def unwrap(cell):
        type_key = next(iter(cell))
        val = cell[type_key]
        if type_key == 'N':
            return Decimal(val)
        return val

    rows = [{k: unwrap(v) for k, v in item.items()} for item in items]
    return pd.DataFrame(rows)


# ── Offices ──────────────────────────────────────────────────────────────────
print("=== DynamoDB: ClassicModels_Offices ===")
df_ddb_offices = scan_table('ClassicModels_Offices')
print(df_ddb_offices.to_string(index=False))


=== DynamoDB: ClassicModels_Offices ===
         city territory addressLine2             addressLine1 officeCode   country            phone postalCode      state
       Boston        NA    Suite 102         1550 Court Place          2       USA  +1 215 837 0825      02107         MA
San Francisco        NA    Suite 300        100 Market Street          1       USA  +1 650 219 4782      94080         CA
       Sydney      APAC     Floor #2    5-11 Wentworth Avenue          6 Australia  +61 2 9264 2451   NSW 2010        NaN
        Tokyo     Japan          NaN              4-1 Kioicho          5     Japan  +81 33 224 5000   102-8578 Chiyoda-Ku
        Paris      EMEA          NaN 43 Rue Jouffroy D'abbans          4    France  +33 14 723 4404      75017        NaN
       London      EMEA      Level 7      25 Old Broad Street          7        UK +44 20 7877 2041   EC2N 1HN        NaN
          NYC        NA      apt. 5A     523 East 53rd Street          3       USA  +1 212 555 3000      1

In [8]:
# ── Customers ────────────────────────────────────────────────────────────────
print("=== DynamoDB: ClassicModels_Customers (first 10) ===")
df_ddb_customers = scan_table('ClassicModels_Customers')
print(df_ddb_customers[['customerNumber','customerName','city','country','creditLimit']].head(10).to_string(index=False))


=== DynamoDB: ClassicModels_Customers (first 10) ===
customerNumber                customerName           city   country creditLimit
           187              AV Stores, Co.     Manchester        UK      136800
           286        Marta's Replicas Co.      Cambridge       USA      123700
           361             Kommission Auto        Münster   Germany           0
           273           Franken Gifts, Co        München   Germany           0
           333 Australian Gift Network, Co South Brisbane Australia       51600
           145    Danish Wholesale Imports      Kobenhavn   Denmark       83400
           256        Auto Associés & Cie.     Versailles    France       77900
           386         L'ordine Souveniers  Reggio Emilia     Italy      121400
           369      Lisboa Souveniers, Inc         Lisboa  Portugal           0
           172   La Corne D'abondance, Co.          Paris    France       84300


## get_item – Fetch a single record by primary key

In RDS you would write `WHERE customerNumber = 103`.  
In DynamoDB you call `get_item` with the exact key — no table scan, constant-time lookup.


In [9]:
response = tbl_customers.get_item(
    Key={'customerNumber': 103}   # partition key value
)
item = response.get('Item')
if item:
    for k, v in sorted(item.items()):
        print(f"  {k:30s} = {v}")
else:
    print("Item not found.")


  addressLine1                   = 54, rue Royale
  city                           = Nantes
  contactFirstName               = Carine 
  contactLastName                = Schmitt
  country                        = France
  creditLimit                    = 100000
  customerName                   = Atelier graphique
  customerNumber                 = 103
  phone                          = 40.32.2555
  postalCode                     = 44000
  salesRepEmployeeNumber         = 1370


## Flexible Schema – Add extra attributes to individual items

In RDS, adding a new column requires `ALTER TABLE`.  
In DynamoDB you simply include new attributes in a single `put_item` / `update_item` call — other rows are unaffected.  
Below we enrich **one customer** with a `preferredContact` and `loyaltyTier` field that no other customer record has.


In [10]:
tbl_customers.update_item(
    Key={'customerNumber': 103},
    UpdateExpression='SET preferredContact = :pc, loyaltyTier = :lt',
    ExpressionAttributeValues={
        ':pc': 'email',
        ':lt': 'Gold',
    },
)

# Read back – only this item has the new attributes
enriched = tbl_customers.get_item(Key={'customerNumber': 103})['Item']
print("Enriched item attributes:")
for k, v in sorted(enriched.items()):
    marker = ' ← NEW' if k in ('preferredContact', 'loyaltyTier') else ''
    print(f"  {k:30s} = {v}{marker}")

# Confirm a different customer does NOT have those attributes
other = tbl_customers.get_item(Key={'customerNumber': 112})['Item']
print(f"\ncustomerNumber 112 has 'loyaltyTier': {'loyaltyTier' in other}")


Enriched item attributes:
  addressLine1                   = 54, rue Royale
  city                           = Nantes
  contactFirstName               = Carine 
  contactLastName                = Schmitt
  country                        = France
  creditLimit                    = 100000
  customerName                   = Atelier graphique
  customerNumber                 = 103
  loyaltyTier                    = Gold ← NEW
  phone                          = 40.32.2555
  postalCode                     = 44000
  preferredContact               = email ← NEW
  salesRepEmployeeNumber         = 1370

customerNumber 112 has 'loyaltyTier': False


## Application-level "JOIN" – Customers enriched with Office data

DynamoDB has no SQL `JOIN`. To relate customers to offices you:
1. Fetch the customer item
2. Look up the related office item using its key
3. Merge in the application layer

This is the DynamoDB way of handling relationships — trade server-side joins for client-side lookups at key-value speed.


In [11]:
# Step 1 – get all customers from DynamoDB (already scanned above)
# Step 2 – get all offices from DynamoDB
df_offices_ddb  = df_ddb_offices.copy()
df_customers_ddb = df_ddb_customers.copy()

# Step 3 – look up salesRepEmployeeNumber → officeCode via RDS employees table
# (DynamoDB has no employees table, so we use RDS for the mapping)
with rds_conn.cursor() as cur:
    cur.execute("SELECT employeeNumber, officeCode FROM employees")
    emp_office_map = {str(r['employeeNumber']): r['officeCode'] for r in cur.fetchall()}

# Step 4 – merge in the application layer
df_customers_ddb['salesRepEmployeeNumber'] = df_customers_ddb['salesRepEmployeeNumber'].astype(str)
df_customers_ddb['officeCode'] = df_customers_ddb['salesRepEmployeeNumber'].map(emp_office_map)

df_enriched = df_customers_ddb.merge(
    df_offices_ddb[['officeCode', 'city']].rename(columns={'city': 'officeCity'}),
    on='officeCode',
    how='left',
)

cols = ['customerNumber', 'customerName', 'country', 'salesRepEmployeeNumber', 'officeCode', 'officeCity']
df_enriched[cols].head(10)


,customerNumber,customerName,country,salesRepEmployeeNumber,officeCode,officeCity
0,187,"AV Stores, Co.",UK,1501,7,London
1,286,Marta's Replicas Co.,USA,1216,2,Boston
2,361,Kommission Auto,Germany,nan,NaN,NaN
3,273,"Franken Gifts, Co",Germany,nan,NaN,NaN
4,333,"Australian Gift Network, Co",Australia,1611,6,Sydney
5,145,Danish Wholesale Imports,Denmark,1401,4,Paris
6,256,Auto Associés & Cie.,France,1370,4,Paris
7,386,L'ordine Souveniers,Italy,1401,4,Paris
8,369,"Lisboa Souveniers, Inc",Portugal,nan,NaN,NaN
9,172,"La Corne D'abondance, Co.",France,1337,4,Paris


## Side-by-side Comparison – RDS vs DynamoDB

Run the same logical question ("list customers in the USA") against both databases.


In [12]:
# ── RDS: SQL WHERE clause ────────────────────────────────────────────────────
with rds_conn.cursor() as cur:
    cur.execute("SELECT customerNumber, customerName, city, creditLimit FROM customers WHERE country = 'USA'")
    rds_usa = cur.fetchall()

df_rds_usa = pd.DataFrame(rds_usa)
print(f"RDS  – USA customers: {len(df_rds_usa)}")
print(df_rds_usa.head(5).to_string(index=False))

print()

# ── DynamoDB: Scan with FilterExpression ────────────────────────────────────
# NOTE: FilterExpression still reads ALL items first, then filters client-side.
# For production use, add a Global Secondary Index on 'country'.
from boto3.dynamodb.conditions import Attr

response = tbl_customers.scan(
    FilterExpression=Attr('country').eq('USA'),
    ProjectionExpression='customerNumber, customerName, city, creditLimit',
)
df_ddb_usa = pd.DataFrame(response['Items'])
print(f"DynamoDB – USA customers: {len(df_ddb_usa)}")
print(df_ddb_usa.sort_values('customerNumber').head(5).to_string(index=False))


RDS  – USA customers: 36
 customerNumber                 customerName          city creditLimit
            112           Signal Gift Stores     Las Vegas    71800.00
            124 Mini Gifts Distributors Ltd.    San Rafael   210500.00
            129              Mini Wheels Co. San Francisco    64600.00
            131            Land of Toys Inc.           NYC   114900.00
            151           Muscle Machine Inc           NYC   138500.00

DynamoDB – USA customers: 36
         city                 customerName creditLimit customerNumber
    Las Vegas           Signal Gift Stores       71800            112
   San Rafael Mini Gifts Distributors Ltd.      210500            124
San Francisco              Mini Wheels Co.       64600            129
          NYC            Land of Toys Inc.      114900            131
          NYC           Muscle Machine Inc      138500            151


## Clean Up

Drop both DynamoDB tables when the demo is done.  
Uncomment and run to avoid ongoing storage costs.


In [13]:
# for table_name in ('ClassicModels_Customers', 'ClassicModels_Offices'):
#     dynamodb.delete_table(TableName=table_name)
#     waiter = dynamodb.get_waiter('table_not_exists')
#     waiter.wait(TableName=table_name)
#     print(f"Table '{table_name}' deleted.")

rds_conn.close()
print("RDS connection closed.")


RDS connection closed.
